# 4. Interpretazione dei Cluster

I cluster sono stati costruiti senza usare label categoriche. Qui le label del dataset vengono usate solo **dopo** il clustering per capire se i gruppi trovati sono leggibili. I nomi sono descrizioni operative post-hoc, non nuove classi astronomiche.

### Nomi descrittivi dei cluster

Sulla base dei profili medi, i 4 cluster vengono denominati:

| Cluster | Nome | Caratteristica principale |
|---|---|---|
| 0 | **Warm Mini-Neptunes** | Pianeti piccoli/leggeri, orbite corte, temperatura catalogata moderata (807 K) |
| 1 | **Hot Jupiters** | Gas giant con temperatura catalogata alta (1227 K), orbite strette |
| 2 | **Ultra-Massive Wide-Orbit Companions** | Compagni estremi: massa molto alta (log 8.1, circa 10-11 masse gioviane), semiasse > 100 AU |
| 3 | **Eccentric Long-Period Giants** | Gas giant con eccentricita' elevata (0.29) e periodi orbitali lunghi |

Questi nomi vengono usati nelle visualizzazioni seguenti per rendere i risultati comunicabili.


In [1]:
from pathlib import Path
import os

if Path.cwd().name != "notebook_final" and (Path.cwd() / "notebook_final").exists():
    os.chdir(Path.cwd() / "notebook_final")

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent
RAW_DATA_PATH = PROJECT_ROOT / "input" / "nasa_exoplanet_intelligence.csv"

import json
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.decomposition import PCA

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

processed_dir = Path("data/processed")
cluster_dir = processed_dir / "clustering"
reports_tables = Path("reports/tables")
reports_figures = Path("reports/figures")
reports_tables.mkdir(parents=True, exist_ok=True)
reports_figures.mkdir(parents=True, exist_ok=True)

df_master = pd.read_csv(processed_dir / "df_master.csv")
X_imputed = pd.read_csv(processed_dir / "X_imputed.csv")
X_clustered = pd.read_csv(cluster_dir / "X_clustered.csv")
with open(cluster_dir / "clustering_metadata.json", encoding="utf-8") as f:
    cluster_meta = json.load(f)

FINAL_FEATURES = cluster_meta["final_features"]
FINAL_K = cluster_meta["final_k"]
FINAL_LABELS = X_clustered["cluster"].values

df_interp = X_imputed[FINAL_FEATURES].copy()
df_interp["cluster"] = FINAL_LABELS
df_labeled = df_master.copy()
df_labeled["cluster"] = FINAL_LABELS

print(f"Configurazione finale: {cluster_meta['final_method']}, k={FINAL_K}, silhouette={cluster_meta['final_silhouette']:.4f}")
display(pd.Series(FINAL_LABELS).value_counts().sort_index().to_frame("n_pianeti"))

CLUSTER_NAMES = {
    0: "Warm Mini-Neptunes",
    1: "Hot Jupiters",
    2: "Ultra-Massive Wide-Orbit Companions",
    3: "Eccentric Long-Period Giants",
}


Configurazione finale: kmeans, k=4, silhouette=0.4471


,n_pianeti
0,4054
1,1174
2,72
3,850


In [2]:
df_raw = pd.read_csv(RAW_DATA_PATH)
if len(df_raw) != len(df_labeled):
    raise ValueError("Il dataset raw e gli artefatti processati non sono allineati")


## 4.1 Profili medi dei cluster


In [3]:
cluster_means = df_interp.groupby("cluster")[FINAL_FEATURES].mean().round(3)
cluster_means.to_csv(reports_tables / "cluster_means_raw.csv")

cluster_means_z = cluster_means.copy()
for col in cluster_means_z.columns:
    std = cluster_means_z[col].std()
    cluster_means_z[col] = (cluster_means_z[col] - cluster_means_z[col].mean()) / std if std > 0 else 0.0
cluster_means_z = cluster_means_z.round(3)
cluster_means_z.to_csv(reports_tables / "cluster_means_zscore.csv")

cluster_means_z_named = cluster_means_z.rename(index=CLUSTER_NAMES)
cluster_means_named = cluster_means.rename(index=CLUSTER_NAMES)

display(cluster_means_named)

plt.figure(figsize=(max(10, len(FINAL_FEATURES) * 1.1), 2 + FINAL_K * 0.9))
sns.heatmap(
    cluster_means_z_named,
    annot=True,
    cmap="coolwarm",
    center=0,
    linewidths=0.5,
    fmt=".2f",
)
plt.title("Profili cluster - z-score delle medie\n(rosso=alto, blu=basso rispetto alla media tra cluster)")
plt.xlabel("Feature")
plt.ylabel("Cluster")
plt.tight_layout()
plt.savefig(reports_figures / "cluster_profile_heatmap.png", dpi=300, bbox_inches="tight")
plt.show()


,equilibrium_temp_k,orbital_eccentricity,orbital_period_days_log,planet_radius_earth_log,planet_mass_earth_log,semi_major_axis_au_log
cluster,,,,,,
Warm Mini-Neptunes,806.765,0.026,2.540,1.160,1.911,0.136
Hot Jupiters,1227.364,0.053,1.935,2.551,5.636,0.257
Ultra-Massive Wide-Orbit Companions,1377.958,0.048,3.586,2.713,8.145,5.773
Eccentric Long-Period Giants,770.849,0.286,6.759,2.542,6.594,1.139


## 4.2 Distribuzioni feature chiave


In [4]:
key_features = [c for c in [
    "orbital_period_days_log",
    "planet_radius_earth_log",
    "planet_mass_earth_log",
    "equilibrium_temp_k",
    "semi_major_axis_au_log",
    "star_temp_k",
    "star_mass_sun",
] if c in FINAL_FEATURES]

n_cols = 3
n_rows = int(np.ceil(len(key_features) / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 4 * n_rows))
axes = np.array(axes).flatten()

for ax, feat in zip(axes, key_features):
    sns.boxplot(data=df_interp, x="cluster", y=feat, palette="tab10", showfliers=False, ax=ax)
    ax.set_title(feat)

    ax.set_xticklabels([CLUSTER_NAMES.get(int(t.get_text()), t.get_text()) for t in ax.get_xticklabels()],
                       rotation=20, ha="right", fontsize=8)
    ax.set_xlabel("")
for ax in axes[len(key_features):]:
    ax.set_visible(False)

plt.suptitle("Distribuzione feature chiave per cluster", y=1.02)
plt.tight_layout()
plt.savefig(reports_figures / "cluster_key_feature_boxplots.png", dpi=300, bbox_inches="tight")
plt.show()


## 4.3 Crosstab post-hoc con label esterne


In [5]:
external_labels = {
    "planet_type": "Blues",
    "star_type": "Greens",
    "habitable_zone_flag": "Oranges",
    "orbital_period_cat": "Purples",
}

for col, cmap in external_labels.items():
    if col not in df_labeled.columns:
        continue
    ct = pd.crosstab(df_labeled["cluster"], df_labeled[col], normalize="index").round(3)

    ct.index = [CLUSTER_NAMES.get(i, str(i)) for i in ct.index]
    ct.to_csv(reports_tables / f"{col}_by_cluster.csv")
    display(ct)

    plt.figure(figsize=(max(8, ct.shape[1] * 1.25), 2 + FINAL_K * 0.7))
    sns.heatmap(ct, annot=True, cmap=cmap, linewidths=0.5, fmt=".2f")
    plt.title(f"{col} per cluster (normalizzato per riga)")
    plt.xlabel(col)
    plt.ylabel("Cluster")
    plt.tight_layout()
    plt.savefig(reports_figures / f"{col}_by_cluster.png", dpi=300, bbox_inches="tight")
    plt.show()


planet_type,Gas Giant,Mini-Neptune,Neptune-like,Sub-Earth,Super-Earth,Super-Jupiter,Unknown
Warm Mini-Neptunes,0.022,0.524,0.106,0.056,0.290,0.000,0.002
Hot Jupiters,0.689,0.012,0.026,0.001,0.005,0.247,0.020
Ultra-Massive Wide-Orbit Companions,0.764,0.000,0.000,0.000,0.000,0.222,0.014
Eccentric Long-Period Giants,0.920,0.013,0.022,0.000,0.005,0.021,0.019


star_type,A-type,B-type,F-type,G-type(Sun-like),K-type,M-type(Red Dwarf),O-type,Unknown
Warm Mini-Neptunes,0.001,0.001,0.154,0.467,0.271,0.086,0.000,0.019
Hot Jupiters,0.009,0.003,0.269,0.371,0.170,0.018,0.000,0.160
Ultra-Massive Wide-Orbit Companions,0.069,0.083,0.111,0.028,0.264,0.333,0.000,0.111
Eccentric Long-Period Giants,0.007,0.005,0.147,0.398,0.385,0.035,0.006,0.018


habitable_zone_flag,0,1
Warm Mini-Neptunes,0.913,0.087
Hot Jupiters,0.996,0.004
Ultra-Massive Wide-Orbit Companions,0.986,0.014
Eccentric Long-Period Giants,0.948,0.052


orbital_period_cat,Long(100-365d),Medium(10-100d),Short(1-10d),Ultra-Short(<1d),Unknown,Very-Long(365d+)
Warm Mini-Neptunes,0.042,0.460,0.442,0.032,0.019,0.004
Hot Jupiters,0.018,0.124,0.675,0.023,0.160,0.001
Ultra-Massive Wide-Orbit Companions,0.000,0.000,0.000,0.000,0.917,0.083
Eccentric Long-Period Giants,0.182,0.085,0.005,0.000,0.002,0.726


## 4.4 Confronto esterno post-hoc: Adjusted Rand Index

L'Adjusted Rand Index (ARI, indicato nella scaletta del corso come Corrected Rand Index) confronta due partizioni correggendo l'accordo atteso per caso. Vale 1 per accordo perfetto, circa 0 per accordo casuale e puo' essere negativo quando l'accordo e' inferiore a quello atteso per caso.

Le label esterne non sono state usate per costruire o selezionare i cluster. Il confronto e' esclusivamente descrittivo: `planet_type` e `orbital_period_cat` derivano in parte da feature incluse e non sono ground truth indipendenti. Per `planet_type` viene esclusa la categoria `Unknown`, che non rappresenta una classe fisica definita.


In [6]:
from sklearn.metrics import adjusted_rand_score

ari_rows = []
for external_label in ["planet_type", "orbital_period_cat", "star_type", "habitable_zone_flag"]:
    if external_label not in df_labeled.columns:
        continue

    valid_mask = df_labeled[external_label].notna()
    if external_label == "planet_type":
        valid_mask &= df_labeled[external_label].ne("Unknown")

    external_values = df_labeled.loc[valid_mask, external_label].astype(str)
    cluster_values = df_labeled.loc[valid_mask, "cluster"]
    ari_rows.append({
        "external_label": external_label,
        "n_samples": int(valid_mask.sum()),
        "n_categories": int(external_values.nunique()),
        "adjusted_rand_index": float(adjusted_rand_score(external_values, cluster_values)),
    })

ari_validation = pd.DataFrame(ari_rows).sort_values("adjusted_rand_index", ascending=False)
ari_validation.to_csv(reports_tables / "cluster_external_validation_ari.csv", index=False)

print("Adjusted Rand Index tra cluster finali e label esterne (solo post-hoc):")
display(ari_validation.round({"adjusted_rand_index": 4}))
print("Interpretazione: accordo parziale con planet_type e orbital_period_cat; accordo vicino o inferiore al caso con star_type e habitable_zone_flag.")


Adjusted Rand Index tra cluster finali e label esterne (solo post-hoc):


,external_label,n_samples,n_categories,adjusted_rand_index
0,planet_type,6100,6,0.3068
1,orbital_period_cat,6150,6,0.2243
2,star_type,6150,8,0.0430
3,habitable_zone_flag,6150,2,-0.0443


Interpretazione: accordo parziale con planet_type e orbital_period_cat; accordo vicino o inferiore al caso con star_type e habitable_zone_flag.


## 4.5 PCA dei cluster finali


In [7]:
X_final = X_clustered.drop(columns=["cluster"])
pca = PCA(n_components=2, random_state=42)
coords = pca.fit_transform(X_final)
pca_df = pd.DataFrame({
    "PC1": coords[:, 0],
    "PC2": coords[:, 1],
    "cluster": FINAL_LABELS,
    "planet_type": df_labeled["planet_type"].values,
})

pca_df["cluster_name"] = pca_df["cluster"].map(CLUSTER_NAMES)

fig, axes = plt.subplots(1, 2, figsize=(18, 6))
sns.scatterplot(data=pca_df, x="PC1", y="PC2", hue="cluster_name",
                palette="tab10", alpha=0.6, s=20, ax=axes[0])
axes[0].set_title("PCA - cluster non supervisionati")
axes[0].legend(title="Cluster", bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8)

sns.scatterplot(data=pca_df, x="PC1", y="PC2", hue="planet_type",
                palette="Set2", alpha=0.6, s=20, ax=axes[1])
axes[1].set_title("PCA - planet_type (solo confronto)")
axes[1].legend(title="planet_type", bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8)

plt.suptitle(f"PCA 2D sul feature set finale - varianza spiegata {pca.explained_variance_ratio_.sum():.1%}")
plt.tight_layout()
plt.savefig(reports_figures / "pca_clusters_vs_planet_type.png", dpi=300, bbox_inches="tight")
plt.show()


## 4.6 Summary tabellare


In [8]:
summary_rows = []
for cid in sorted(np.unique(FINAL_LABELS)):
    mask = df_labeled["cluster"] == cid
    subset = df_labeled.loc[mask]
    row = {
        "cluster": int(cid),
        "cluster_name": CLUSTER_NAMES.get(int(cid), f"Cluster {cid}"),
        "size": int(mask.sum()),
        "pct_total": round(mask.sum() / len(df_labeled) * 100, 2),
        "top_planet_type": subset["planet_type"].mode(dropna=True).iloc[0],
        "top_star_type": subset["star_type"].mode(dropna=True).iloc[0],
        "habitable_zone_rate": round(subset["habitable_zone_flag"].mean(), 3),
    }
    for feat in FINAL_FEATURES:
        row[f"{feat}_mean"] = round(df_interp.loc[df_interp["cluster"] == cid, feat].mean(), 3)
    summary_rows.append(row)

cluster_summary = pd.DataFrame(summary_rows)
cluster_summary.to_csv(reports_tables / "cluster_summary.csv", index=False)

print("Profilo sintetico dei cluster:")
display(cluster_summary[["cluster", "cluster_name", "size", "pct_total",
                           "top_planet_type", "top_star_type", "habitable_zone_rate"]])


Profilo sintetico dei cluster:


,cluster,cluster_name,size,pct_total,top_planet_type,top_star_type,habitable_zone_rate
0,0,Warm Mini-Neptunes,4054,65.92,Mini-Neptune,G-type(Sun-like),0.087
1,1,Hot Jupiters,1174,19.09,Gas Giant,G-type(Sun-like),0.004
2,2,Ultra-Massive Wide-Orbit Companions,72,1.17,Gas Giant,M-type(Red Dwarf),0.014
3,3,Eccentric Long-Period Giants,850,13.82,Gas Giant,G-type(Sun-like),0.052


## 4.7 Audit del cluster estremo e sensitivity analysis

Il cluster 2 richiede un controllo specifico perche' combina semiassi molto ampi con forte missingness del periodo orbitale. L'audit seguente torna ai valori raw, quantifica le imputazioni e confronta i periodi disponibili con una previsione approssimata della terza legge di Keplero.

La colonna `equilibrium_temp_k` viene trattata come **temperatura catalogata**: per molti compagni scoperti tramite direct imaging i valori sono incompatibili con la sola temperatura di equilibrio irradiativo e possono riflettere temperature effettive di oggetti giovani auto-luminosi. La stima irradiativa usata qui e' solo diagnostica e assume albedo zero e redistribuzione completa.

Infine, una sensitivity analysis post-hoc rifitta K-Means `k=4` rimuovendo periodo, temperatura o entrambe. Non e' un criterio di selezione ne' una validazione indipendente: verifica soltanto se il piccolo gruppo estremo dipenda esclusivamente dalle due feature problematiche.


In [9]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

cluster2_mask = df_labeled["cluster"].eq(2)
raw_cluster2 = df_raw.loc[cluster2_mask].copy()

period_valid = (
    raw_cluster2["orbital_period_days"].notna()
    & raw_cluster2["semi_major_axis_au"].notna()
    & raw_cluster2["star_mass_sun"].gt(0)
)
period_subset = raw_cluster2.loc[period_valid]
expected_period_days = 365.25 * np.sqrt(
    period_subset["semi_major_axis_au"].pow(3) / period_subset["star_mass_sun"]
)
period_ratio = period_subset["orbital_period_days"] / expected_period_days

temp_valid = (
    raw_cluster2["equilibrium_temp_k"].notna()
    & raw_cluster2["semi_major_axis_au"].gt(0)
    & raw_cluster2["star_temp_k"].gt(0)
    & raw_cluster2["star_radius_sun"].gt(0)
)
temp_subset = raw_cluster2.loc[temp_valid]
expected_irradiative_temp = temp_subset["star_temp_k"] * np.sqrt(
    (temp_subset["star_radius_sun"] * 0.00465047)
    / (2 * temp_subset["semi_major_axis_au"])
)
temperature_ratio = temp_subset["equilibrium_temp_k"] / expected_irradiative_temp

observed_periods = raw_cluster2["orbital_period_days"].dropna()
audit = pd.DataFrame([{
    "cluster": 2,
    "cluster_name": CLUSTER_NAMES[2],
    "cluster_size": int(len(raw_cluster2)),
    "direct_imaging_count": int(raw_cluster2["discovery_method"].eq("Imaging").sum()),
    "orbital_period_missing": int(raw_cluster2["orbital_period_days"].isna().sum()),
    "catalogued_temperature_missing": int(raw_cluster2["equilibrium_temp_k"].isna().sum()),
    "semi_major_axis_missing": int(raw_cluster2["semi_major_axis_au"].isna().sum()),
    "semi_major_axis_median_au": float(raw_cluster2["semi_major_axis_au"].median()),
    "semi_major_axis_min_au": float(raw_cluster2["semi_major_axis_au"].min()),
    "semi_major_axis_max_au": float(raw_cluster2["semi_major_axis_au"].max()),
    "observed_period_count": int(len(observed_periods)),
    "observed_period_median_days": float(observed_periods.median()),
    "observed_period_geometric_mean_days": float(np.expm1(np.log1p(observed_periods).mean())),
    "kepler_complete_count": int(len(period_ratio)),
    "observed_over_kepler_period_median": float(period_ratio.median()),
    "temperature_physics_complete_count": int(len(temperature_ratio)),
    "catalogued_temperature_median_k": float(temp_subset["equilibrium_temp_k"].median()),
    "irradiative_temperature_median_k": float(expected_irradiative_temp.median()),
    "catalogued_over_irradiative_temp_median": float(temperature_ratio.median()),
}])
audit.to_csv(reports_tables / "cluster2_data_quality_audit.csv", index=False)
display(audit.T.rename(columns={0: "value"}))

set_a_scaled = X_clustered.drop(columns=["cluster"])
sensitivity_cases = {
    "full_set_A": list(set_a_scaled.columns),
    "minus_period": [c for c in set_a_scaled.columns if c != "orbital_period_days_log"],
    "minus_catalogued_temperature": [c for c in set_a_scaled.columns if c != "equilibrium_temp_k"],
    "minus_period_and_catalogued_temperature": [
        c for c in set_a_scaled.columns
        if c not in {"orbital_period_days_log", "equilibrium_temp_k"}
    ],
}

original_labels = X_clustered["cluster"].to_numpy()
original_extreme = original_labels == 2
sensitivity_rows = []
for case_name, case_features in sensitivity_cases.items():
    case_labels = KMeans(
        n_clusters=FINAL_K, random_state=42, n_init=20
    ).fit_predict(set_a_scaled[case_features])

    matches = []
    for candidate_cluster in np.unique(case_labels):
        candidate_mask = case_labels == candidate_cluster
        intersection = int(np.sum(candidate_mask & original_extreme))
        union = int(np.sum(candidate_mask | original_extreme))
        matches.append({
            "matched_cluster": int(candidate_cluster),
            "overlap_with_original_cluster2": intersection,
            "matched_cluster_size": int(candidate_mask.sum()),
            "jaccard_with_original_cluster2": intersection / union,
        })
    best_match = max(matches, key=lambda row: row["jaccard_with_original_cluster2"])
    sensitivity_rows.append({
        "case": case_name,
        "n_features": len(case_features),
        "features": "|".join(case_features),
        "silhouette": float(silhouette_score(set_a_scaled[case_features], case_labels)),
        "ari_vs_final_partition": float(adjusted_rand_score(original_labels, case_labels)),
        **best_match,
    })

sensitivity = pd.DataFrame(sensitivity_rows)
sensitivity.to_csv(reports_tables / "cluster2_sensitivity_analysis.csv", index=False)
display(sensitivity.round(4))
print("La sensitivity analysis risulta post-hoc: descrive robustezza, non seleziona una nuova soluzione finale.")


,value
cluster,2
cluster_name,Ultra-Massive Wide-Orbit Companions
cluster_size,72
direct_imaging_count,69
orbital_period_missing,66
catalogued_temperature_missing,28
semi_major_axis_missing,0
semi_major_axis_median_au,325.0
semi_major_axis_min_au,22.3
semi_major_axis_max_au,19000.0


,case,n_features,features,silhouette,ari_vs_final_partition,matched_cluster,overlap_with_original_cluster2,matched_cluster_size,jaccard_with_original_cluster2
0,full_set_A,6,equilibrium_temp_k|orbital_eccentricity|orbita...,0.4471,1.0000,2,72,72,1.0000
1,minus_period,5,equilibrium_temp_k|orbital_eccentricity|planet...,0.4741,0.8731,3,72,80,0.9000
2,minus_catalogued_temperature,5,orbital_eccentricity|orbital_period_days_log|p...,0.5099,0.9244,3,72,72,1.0000
3,minus_period_and_catalogued_temperature,4,orbital_eccentricity|planet_radius_earth_log|p...,0.5907,0.8335,3,72,81,0.8889


La sensitivity analysis risulta post-hoc: descrive robustezza, non seleziona una nuova soluzione finale.


## 4.8 Interpretazione narrativa dei cluster

I quattro gruppi sono partizioni esplorative interpretabili, non classi astronomiche definitive:

- **Cluster 0 — Warm Mini-Neptunes** (65.9%): gruppo dominante con massa e raggio ridotti, temperatura catalogata moderata (~807 K) e orbite compatte.

- **Cluster 1 — Hot Jupiters** (19.1%): gas giant con temperatura catalogata elevata (~1227 K), massa/raggio maggiori e orbite strette.

- **Cluster 2 — Ultra-Massive Wide-Orbit Companions** (1.2%): piccolo gruppo estremo di 72 oggetti. La media logaritmica della massa corrisponde a circa 11 masse gioviane e il semiasse mediano raw e' circa 325 AU. Sessantanove oggetti sono stati scoperti tramite direct imaging; massa, distanza orbitale e regime osservativo li collocano vicino alla zona di ambiguita' pianeta/oggetto substellare.

  Il periodo medio trasformato non rappresenta la scala fisica del gruppo, perche' 66 valori su 72 sono imputati. I sei periodi osservati sono invece dell'ordine di milioni di giorni e risultano approssimativamente coerenti con la terza legge di Keplero. Anche la temperatura catalogata richiede cautela: nei compagni auto-luminosi puo' riflettere una temperatura effettiva piu' che l'equilibrio irradiativo. La sensitivity analysis post-hoc mantiene tutti i 72 oggetti nello stesso gruppo abbinato anche rimuovendo periodo e temperatura, ma non trasforma il cluster in una classe fisica validata.

- **Cluster 3 — Eccentric Long-Period Giants** (13.8%): gas giant con eccentricita' media 0.29 e periodo logaritmico medio 6.76, equivalente a circa 861 giorni.

> **Nota metodologica**: nomi e interpretazioni sono post-hoc. Non sono stati usati nel fit o nella selezione. Le label esterne e la sensitivity analysis descrivono il risultato, ma non costituiscono ground truth indipendente.


In [10]:
print("Riepilogo comparativo dei cluster:\n")
cols_show = ["cluster_name", "size", "pct_total", "top_planet_type",
             "habitable_zone_rate",
             "equilibrium_temp_k_mean", "orbital_eccentricity_mean",
             "planet_mass_earth_log_mean", "semi_major_axis_au_log_mean"]
cols_show = [c for c in cols_show if c in cluster_summary.columns]
display(cluster_summary[cols_show].set_index("cluster_name"))


Riepilogo comparativo dei cluster:



,size,pct_total,top_planet_type,habitable_zone_rate,equilibrium_temp_k_mean,orbital_eccentricity_mean,planet_mass_earth_log_mean,semi_major_axis_au_log_mean
cluster_name,,,,,,,,
Warm Mini-Neptunes,4054,65.92,Mini-Neptune,0.087,806.765,0.026,1.911,0.136
Hot Jupiters,1174,19.09,Gas Giant,0.004,1227.364,0.053,5.636,0.257
Ultra-Massive Wide-Orbit Companions,72,1.17,Gas Giant,0.014,1377.958,0.048,8.145,5.773
Eccentric Long-Period Giants,850,13.82,Gas Giant,0.052,770.849,0.286,6.594,1.139
